In [2]:
#Text_cleaning_and_normalization_tool(Phue_Pwint)


import re
import difflib
import gradio as gr
import nltk
from spellchecker import SpellChecker
from difflib import SequenceMatcher
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

spell = SpellChecker()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def remove_extra_spaces(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,!?])", r"\1", text)
    return text.strip()



def remove_consecutive_duplicates(text):
    tokens = re.findall(r"\w+|[^\w\s]", text)
    new_tokens = []
    prev_word = None

    for token in tokens:
        if token.isalpha():
            word = token.lower()

            if prev_word:
                similarity = difflib.SequenceMatcher(None, word, prev_word).ratio()
            else:
                similarity = 0

            if similarity < 0.85 or word in {"i", "a"}:
                new_tokens.append(token)

            prev_word = word
        else:
            new_tokens.append(token)

    return " ".join(new_tokens)

def spelling_correct(text):
    words = text.split()
    corrected = []

    for w in words:
        if w.lower() == "i":
            corrected.append("I")
        else:
            suggestion = spell.correction(w)
           
            if suggestion and difflib.SequenceMatcher(None, w.lower(), suggestion.lower()).ratio() > 0.7:
                corrected.append(suggestion)
            else:
                corrected.append(w)
    return " ".join(corrected)

def remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w.lower() not in stop_words]
    return " ".join(filtered)

from nltk.corpus import wordnet
from nltk import pos_tag

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
        
def lemmatize_text(text):
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)

    lemmas = []
    for word, tag in pos_tags:
        wn_tag = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(word, wn_tag)
        lemmas.append(lemma)

    return " ".join(lemmas)


def ocr_char_correction(text):
    
    
    text = re.sub(r'(?<=\w)0(?=\w)', 'O', text)
    text = re.sub(r'(?<=\w)1(?=\w)', 'l', text)
    text = text.replace("rn", "m")
    text = text.replace("vv", "w")
    return text

def full_normalize(text, use_stopwords=False, use_lemma=False):
   
    if not text:
        return ""
    
    text = remove_extra_spaces(text)
    
    text = text.replace("0", "o").replace("1", "l").replace("3", "e")
    
   
    text = ocr_char_correction(text)
   
    text = remove_consecutive_duplicates(text)
    
   
    if use_stopwords:
        text = remove_stopwords(text)
    
    
    text = spelling_correct(text)
    
    
    if use_lemma:
        text = lemmatize_text(text)
    
    
    if text:
        text = text[0].upper() + text[1:]
    
    return text



def smart_tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text, re.UNICODE)

def align_words(original, corrected):
    orig_tokens = smart_tokenize(original)
    corr_tokens = smart_tokenize(corrected)
    matcher = SequenceMatcher(None, orig_tokens, corr_tokens)

    highlighted = ""
    errors = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            highlighted += " ".join(orig_tokens[i1:i2]) + " "
        elif tag == "replace":
            for o, c in zip(orig_tokens[i1:i2], corr_tokens[j1:j2]):
                highlighted += f"<span style='background:#ffb3b3'>{o}</span> "
                highlighted += f"<span style='background:#b3ffb3'>{c}</span> "
                errors.append([o, c])
        elif tag == "delete":
            for o in orig_tokens[i1:i2]:
                highlighted += f"<span style='background:#ffb3b3'>{o}</span> "
                errors.append([o, ""])
        elif tag == "insert":
            for c in corr_tokens[j1:j2]:
                highlighted += f"<span style='background:#b3ffb3'>{c}</span> "
                errors.append(["", c])

    return highlighted.strip(), errors[:5]

def calculate_cer(original, corrected):
    sm = SequenceMatcher(None, original, corrected)
    matches = sum(triple.size for triple in sm.get_matching_blocks())
    if len(original) == 0:
        return 0
    return round(1 - (matches / len(original)), 4)

def calculate_wer(original, corrected):
    o_words = remove_extra_spaces(original).split()
    c_words = remove_extra_spaces(corrected).split()
    sm = SequenceMatcher(None, o_words, c_words)
    matches = sum(triple.size for triple in sm.get_matching_blocks())
    if len(o_words) == 0:
        return 0
    return round(1 - (matches / len(o_words)), 4)

def confidence_score(cer):
    return round(max(0, 1 - cer) * 100, 2)


def process(text, use_stopwords, use_lemma):
    if not text:
        return "", [], "", 0, 0, 0

    corrected = full_normalize(text, use_stopwords, use_lemma)
    highlighted, table_data = align_words(text, corrected)

    cer = calculate_cer(text, corrected)
    wer = calculate_wer(text, corrected)
    conf = confidence_score(cer)

    return highlighted, table_data, corrected, cer, wer, conf


custom_css = """
/* Background Gradient */
body {
    background: linear-gradient(145deg, #4facfe, #00f2fe);
    font-family: 'Roboto', 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
}

/* Card Section Style */
.card {
    background: white;
    padding: 20px;
    border-radius: 12px;
    box-shadow: 0px 6px 18px rgba(0,0,0,0.15);
    margin-bottom: 15px;
}

/* Metric Box Style */
.metric-box {
    background: #f0f8ff;
    padding: 15px;
    border-radius: 10px;
    text-align: center;
    font-weight: bold;
    font-size: 16px;
}

/* Markdown & Text Elements */
.gr-markdown, .gr-textbox, .gr-number, .gr-html {
    font-family: 'Roboto', 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
}
"""

with gr.Blocks(
    title="Advanced Text Cleaning Tool",
    css=custom_css,
    theme=gr.themes.Soft(primary_hue="cyan")
) as demo:

    gr.Markdown("# Text Cleaning & Normalization")
   

    with gr.Tabs():
        with gr.Tab("📝 Text Cleaning Tool"):
            with gr.Row():
                with gr.Column(scale=2):
                    with gr.Group(elem_classes="card"):
                        input_box = gr.Textbox(label="📄 Raw Text", lines=8)
                        stop_toggle = gr.Checkbox(label="Remove Stopwords")
                        lemma_toggle = gr.Checkbox(label="Apply Lemmatization")
                        btn = gr.Button("🚀 Process", variant="primary")

                with gr.Column(scale=1):
                    with gr.Group(elem_classes="card"):
                        cer_box = gr.Number(label="CER")
                        wer_box = gr.Number(label="WER")
                        conf_box = gr.Number(label="Confidence (%)")

            with gr.Group(elem_classes="card"):
                highlight_out = gr.HTML(label="🔍 Highlighted Changes")
                table_out = gr.Dataframe(headers=["Wrong", "Correct"])
                clean_out = gr.Textbox(label="✅ Final Output", lines=5)


        with gr.Tab("📚 About us"):
            with gr.Group(elem_classes="card"):
                gr.Markdown("""
## 📖 အသုံးပြုထားသော Libraries များ

### Project Information
- **Project Title:** Text Cleaning and Normalization Tool for Raw Documents
- **Specialization:** Natural Language Processing (NLP)
- **University:** University of Computer Studies, Pyay

**Developer:**  
- Name: Ma Phue Pwint Khine  
- Roll No: PaKaPaTa-002054  
- Year: Final Year

### 1️⃣ re (Regular Expression)
- OCR error ပြင်ဆင်ခြင်း  
- Extra space ဖယ်ရှားခြင်း  
- Text format ပြုပြင်ခြင်း  

### 2️⃣ difflib
- Original & Corrected text နှိုင်းယှဉ်ခြင်း  
- Highlight error ပြခြင်း  
- CER / WER တွက်ချက်ခြင်း  

### 3️⃣ gradio
- Web UI တည်ဆောက်ခြင်း  
- Tabs, Button, Output ပြခြင်း  

### 4️⃣ nltk
- Tokenization  
- Stopwords Removal  
- Lemmatization  
- POS tagging

### 5️⃣ spellchecker
- OCR spelling error အလိုအလျောက် ပြင်ဆင်ခြင်း  

---

## 🎯 Project Summary
ဤစနစ်သည် RawText များကို  
**Text Cleaning + NLP Techniques** အသုံးပြု၍ Error Detection, Normalization, Accuracy Metrics  
ပေးနိုင်သော AI-based Tool ဖြစ်ပါသည်။
""")


    btn.click(
        process,
        inputs=[input_box, stop_toggle, lemma_toggle],
        outputs=[highlight_out, table_out, clean_out, cer_box, wer_box, conf_box]
    )

demo.launch()

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
C:\Users\ASUS\AppData\Local\Temp\ipykernel_25304\2001024570.py:241: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
